In [1]:
# ============================================
# Appointment Dataset NLP + K-Means Clustering
# ============================================

# Install libraries (run once if needed)
# !pip install pandas numpy scikit-learn nltk matplotlib seaborn

import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt


# Download NLP resources
nltk.download('punkt')
nltk.download('stopwords')


# ============================================
# 1. Load Dataset
# ============================================

df = pd.read_csv("appointments.csv")

# Display dataset
df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,appointment_id,judge_id,institution_name,position_president,position_chamber_chair,open_call_number,appointing_cabinet_id,appointing_cabinet_name,appointment_date,period_start_year,...,publ_agency,publ_omb,publ_pros,publ_int,just_dep,fin_dep,other_dep,prep_one,prep_multi,honor_doc
0,1,323,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,5,7,0,0,1,0,0,1,1,1
1,2,322,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,0,0,0,0,1,0,0,1,1,1
2,3,321,Högsta domstolen,0,0,12-2018,1149,Lofven I,2018-06-14,2018,...,0,0,0,0,0,0,0,0,0,0
3,4,320,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,0,0,0
4,5,319,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,1,1,1


In [2]:
df.info()

print(df.columns)

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 56 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   appointment_id           545 non-null    int64  
 1   judge_id                 545 non-null    int64  
 2   institution_name         545 non-null    str    
 3   position_president       545 non-null    int64  
 4   position_chamber_chair   545 non-null    int64  
 5   open_call_number         545 non-null    str    
 6   appointing_cabinet_id    545 non-null    int64  
 7   appointing_cabinet_name  355 non-null    str    
 8   appointment_date         327 non-null    str    
 9   period_start_year        545 non-null    int64  
 10  period_end_year          545 non-null    int64  
 11  period_end_date          12 non-null     str    
 12  reason_leaving           532 non-null    float64
 13  last_name                545 non-null    str    
 14  first_name               545 non-null

In [4]:
df.columns

Index(['appointment_id', 'judge_id', 'institution_name', 'position_president',
       'position_chamber_chair', 'open_call_number', 'appointing_cabinet_id',
       'appointing_cabinet_name', 'appointment_date', 'period_start_year',
       'period_end_year', 'period_end_date', 'reason_leaving', 'last_name',
       'first_name', 'middle_name', 'previous_appointments', 'female',
       'birth_year', 'degree_year', 'degree_university', 'acad_lic',
       'acad_doctor', 'acad_lekt', 'acad_docent', 'acad_prof', 'dep_spec',
       'dep_hand', 'dep_rad', 'dep_chef', 'ct_edu', 'ct_tr', 'ct_fr',
       'ct_hovr', 'ct_kamr', 'ct_int', 'ct_other', 'ct_clerk', 'priv_assoc',
       'priv_adv', 'priv_corp', 'pol_empl', 'pol_other', 'pol_parl', 'pol_min',
       'publ_amb', 'publ_agency', 'publ_omb', 'publ_pros', 'publ_int',
       'just_dep', 'fin_dep', 'other_dep', 'prep_one', 'prep_multi',
       'honor_doc'],
      dtype='str')

In [5]:

df.head()

,appointment_id,judge_id,institution_name,position_president,position_chamber_chair,open_call_number,appointing_cabinet_id,appointing_cabinet_name,appointment_date,period_start_year,...,publ_agency,publ_omb,publ_pros,publ_int,just_dep,fin_dep,other_dep,prep_one,prep_multi,honor_doc
0,1,323,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,5,7,0,0,1,0,0,1,1,1
1,2,322,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,0,0,0,0,1,0,0,1,1,1
2,3,321,Högsta domstolen,0,0,12-2018,1149,Lofven I,2018-06-14,2018,...,0,0,0,0,0,0,0,0,0,0
3,4,320,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,0,0,0
4,5,319,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,1,1,1


In [6]:

df.head()

,appointment_id,judge_id,institution_name,position_president,position_chamber_chair,open_call_number,appointing_cabinet_id,appointing_cabinet_name,appointment_date,period_start_year,...,publ_agency,publ_omb,publ_pros,publ_int,just_dep,fin_dep,other_dep,prep_one,prep_multi,honor_doc
0,1,323,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,5,7,0,0,1,0,0,1,1,1
1,2,322,Högsta domstolen,0,0,75-2018,1587,Lofven II,2018-12-20,2019,...,0,0,0,0,1,0,0,1,1,1
2,3,321,Högsta domstolen,0,0,12-2018,1149,Lofven I,2018-06-14,2018,...,0,0,0,0,0,0,0,0,0,0
3,4,320,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,0,0,0
4,5,319,Högsta domstolen,0,0,5-2016,1149,Lofven I,2016-09-15,2017,...,0,0,0,0,1,0,0,1,1,1


In [8]:
print(df.columns)

Index(['appointment_id', 'judge_id', 'institution_name', 'position_president',
       'position_chamber_chair', 'open_call_number', 'appointing_cabinet_id',
       'appointing_cabinet_name', 'appointment_date', 'period_start_year',
       'period_end_year', 'period_end_date', 'reason_leaving', 'last_name',
       'first_name', 'middle_name', 'previous_appointments', 'female',
       'birth_year', 'degree_year', 'degree_university', 'acad_lic',
       'acad_doctor', 'acad_lekt', 'acad_docent', 'acad_prof', 'dep_spec',
       'dep_hand', 'dep_rad', 'dep_chef', 'ct_edu', 'ct_tr', 'ct_fr',
       'ct_hovr', 'ct_kamr', 'ct_int', 'ct_other', 'ct_clerk', 'priv_assoc',
       'priv_adv', 'priv_corp', 'pol_empl', 'pol_other', 'pol_parl', 'pol_min',
       'publ_amb', 'publ_agency', 'publ_omb', 'publ_pros', 'publ_int',
       'just_dep', 'fin_dep', 'other_dep', 'prep_one', 'prep_multi',
       'honor_doc'],
      dtype='str')


In [9]:
print(df.columns.tolist())

['appointment_id', 'judge_id', 'institution_name', 'position_president', 'position_chamber_chair', 'open_call_number', 'appointing_cabinet_id', 'appointing_cabinet_name', 'appointment_date', 'period_start_year', 'period_end_year', 'period_end_date', 'reason_leaving', 'last_name', 'first_name', 'middle_name', 'previous_appointments', 'female', 'birth_year', 'degree_year', 'degree_university', 'acad_lic', 'acad_doctor', 'acad_lekt', 'acad_docent', 'acad_prof', 'dep_spec', 'dep_hand', 'dep_rad', 'dep_chef', 'ct_edu', 'ct_tr', 'ct_fr', 'ct_hovr', 'ct_kamr', 'ct_int', 'ct_other', 'ct_clerk', 'priv_assoc', 'priv_adv', 'priv_corp', 'pol_empl', 'pol_other', 'pol_parl', 'pol_min', 'publ_amb', 'publ_agency', 'publ_omb', 'publ_pros', 'publ_int', 'just_dep', 'fin_dep', 'other_dep', 'prep_one', 'prep_multi', 'honor_doc']


In [10]:
print(df.columns.tolist())

['appointment_id', 'judge_id', 'institution_name', 'position_president', 'position_chamber_chair', 'open_call_number', 'appointing_cabinet_id', 'appointing_cabinet_name', 'appointment_date', 'period_start_year', 'period_end_year', 'period_end_date', 'reason_leaving', 'last_name', 'first_name', 'middle_name', 'previous_appointments', 'female', 'birth_year', 'degree_year', 'degree_university', 'acad_lic', 'acad_doctor', 'acad_lekt', 'acad_docent', 'acad_prof', 'dep_spec', 'dep_hand', 'dep_rad', 'dep_chef', 'ct_edu', 'ct_tr', 'ct_fr', 'ct_hovr', 'ct_kamr', 'ct_int', 'ct_other', 'ct_clerk', 'priv_assoc', 'priv_adv', 'priv_corp', 'pol_empl', 'pol_other', 'pol_parl', 'pol_min', 'publ_amb', 'publ_agency', 'publ_omb', 'publ_pros', 'publ_int', 'just_dep', 'fin_dep', 'other_dep', 'prep_one', 'prep_multi', 'honor_doc']


In [11]:
df.to_csv(
    "appointments_clustered.csv",
    index=False
)

print("Saved successfully")

Saved successfully
